In [1]:
!pip install numpy==1.23.5 scipy==1.10.1 gensim==4.3.1 --quiet

# Finding similar documents using doc2vec

In [4]:
import warnings
warnings.filterwarnings('ignore')
import gensim
from gensim.models.doc2vec import TaggedDocument
from nltk import RegexpTokenizer
tokenizer = RegexpTokenizer(r'\w+')
import nltk
nltk.download('stopwords')
stopWords = set(stopwords.words('english'))
import os

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
import zipfile
import os

# Unzip 'news_dataset.zip' into the 'data' directory
with zipfile.ZipFile('news_dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

with zipfile.ZipFile('news_dataset.zip', 'r') as zip_ref:
    zip_ref.printdir()


File Name                                             Modified             Size
news_dataset/                                  2025-05-30 20:10:10            0
news_dataset/Electronics_0.txt                 2025-05-30 20:09:10         2347
news_dataset/Electronics_1.txt                 2025-05-30 20:09:10         1607
news_dataset/Electronics_10.txt                2025-05-30 20:09:10          809
news_dataset/Electronics_100.txt               2025-05-30 20:09:10         1572
news_dataset/Electronics_101.txt               2025-05-30 20:09:10         2187
news_dataset/Electronics_102.txt               2025-05-30 20:09:10         1512
news_dataset/Electronics_103.txt               2025-05-30 20:09:10         1291
news_dataset/Electronics_104.txt               2025-05-30 20:09:10          545
news_dataset/Electronics_105.txt               2025-05-30 20:09:10         1167
news_dataset/Electronics_106.txt               2025-05-30 20:09:10         1155
news_dataset/Electronics_107.txt        

In [8]:
docLabels = [f for f in os.listdir('data/news_dataset') if f.endswith('.txt')]
print(f"Found {len(docLabels)} text files.")
data = []
for doc in docLabels:
  data.append(open('data/news_dataset/'+doc).read())

Found 41 text files.


In [9]:
docLabels[:5]

['Electronics_109.txt',
 'Electronics_128.txt',
 'Electronics_133.txt',
 'Electronics_114.txt',
 'Electronics_104.txt']

In [10]:
class DocIterator(object):
  def __init__(self, doc_list, labels_list):
    self.labels_list = labels_list
    self.doc_list = doc_list
  def __iter__(self):
    for idx, doc in enumerate(self.doc_list):
      yield TaggedDocument(words=doc.split(), tags=[self.labels_list[idx]])

In [11]:
it = DocIterator(data, docLabels)

In [12]:
size = 100
alpha = 0.025
min_alpha = 0.025
dm = 1
min_count = 1

In [15]:
model = gensim.models.Doc2Vec(min_count=min_count, alpha=alpha,min_alpha=min_alpha, dm=dm, vector_size=size)
model.build_vocab(it)

In [20]:
epochs = 100
alpha_start = 0.025
alpha_end = 0.0001

alpha_delta = (alpha_start - alpha_end) / epochs
alpha = alpha_start

for epoch in range(epochs):
    model.train(it, total_examples=120, epochs=1)
    alpha -= alpha_delta
    model.alpha = alpha
    model.min_alpha = alpha


In [21]:
model.save('/content/doc2vec.model')

In [22]:
d2v_model = gensim.models.doc2vec.Doc2Vec.load('/content/doc2vec.model')

In [24]:
d2v_model.docvecs.most_similar('Electronics_0.txt')

[('Electronics_108.txt', 0.8873727321624756),
 ('Electronics_131.txt', 0.8653790354728699),
 ('Electronics_115.txt', 0.8568223714828491),
 ('Electronics_109.txt', 0.8536149859428406),
 ('Electronics_113.txt', 0.8471747040748596),
 ('Electronics_101.txt', 0.832639217376709),
 ('Electronics_114.txt', 0.8317624926567078),
 ('Electronics_133.txt', 0.8269955515861511),
 ('Electronics_13.txt', 0.8269559741020203),
 ('Electronics_107.txt', 0.8262884616851807)]

# Understanding skip-thoughts algorithm

Skip-thoughts is one of the popular unsupervised learning algorithms for
learning the sentence embedding. We can see skip-thoughts as an analogy to
the skip-gram model. We learned that in the skip-gram model, we try to
predict the context word given a target word, whereas in skip-thoughts, we try
to predict the context sentence given a target sentence. In other words, we can
say that skip-gram is used for learning word-level vectors and skip-thoughts
is used for learning sentence-level vectors.

The algorithm of skip-thoughts is very simple. It consists of an encoder-
decoder architecture. The role of the encoder is to map the sentence to a vector and the role of the decoder is to generate the surrounding sentences that
is the previous and next sentence of the given input sentence.

# Quick-thoughts for sentence embeddings

Quick-thoughts is another interesting algorithm for learning the sentence
embeddings. In skip-thoughts, we saw how we used the encoder-decoder
architecture to learn the sentence embeddings. In quick-thoughts, we try to
learn whether a given sentence is related to the candidate sentence. So,
instead of using a decoder, we use a classifier to learn whether a given input
sentence is related to the candidate sentence.